<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/6_1_Kernel_Trick%2C_Mercer's_Theorem%2C_and_Kernel_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VI. Kernel Methods

## Kernel Trick, Mercer's Theorem, and Kernel Methods

## Kernel PCA and Kernel Logistic Regression

## Gaussian Processes

### От линейных методов к ядровому трюку: мотивация и отображение в пространство признаков

Линейные методы — SVM, PCA, логистическая регрессия — мощны и хорошо изучены, однако их способность разделять данные ограничена линейными поверхностями. В бесчисленных прикладных задачах классы не могут быть разделены прямой или гиперплоскостью: типичный пример — данные, сгруппированные в виде концентрических окружностей или «полумесяцев». Один из элегантных способов справиться с нелинейностью состоит в том, чтобы отобразить исходные векторы в новое, более богатое пространство признаков, где линейное разделение становится возможным. Однако прямое построение такого отображения может быть вычислительно неподъёмным или даже невозможным — особенно если размерность целевого пространства огромна или бесконечна. **Ядровой трюк (kernel trick)** позволяет обойти явное вычисление новых координат, работая исключительно со скалярными произведениями в том пространстве. Такой подход не только даёт вычислительную эффективность, но и открывает доступ к бесконечномерным гильбертовым пространствам, не требуя их явного конструирования.

---

#### 1.1. Мотивация: от нелинейности к линейности

Рассмотрим задачу бинарной классификации на плоскости $\mathcal{X} = \mathbb{R}^2$. Пусть объекты класса $+1$ сосредоточены внутри единичной окружности $x_1^2 + x_2^2 < 1$, а объекты класса $-1$ — вне её. Никакая прямая не может правильно разделить эти два множества. Однако, если мы построим отображение

$$
\phi : \mathbb{R}^2 \to \mathbb{R}^3, \qquad
\phi(\mathbf{x}) = \phi(x_1, x_2) = \begin{pmatrix} x_1 \\ x_2 \\ x_1^2 + x_2^2 \end{pmatrix},
$$

данные становятся линейно разделимыми в трёхмерном пространстве: плоскость $z = 1$ отделяет точки, для которых $x_1^2 + x_2^2 < 1$, от точек с $x_1^2 + x_2^2 > 1$. Таким образом, нелинейная задача на плоскости свелась к линейной в пространстве более высокой размерности. Это пространство часто называют **спрямляющим**: в нём сложная нелинейная граница становится линейной.

Этот простой пример иллюстрирует общую стратегию: выбрать отображение $\phi : \mathcal{X} \to \mathcal{H}$ из исходного пространства $\mathcal{X}$ в некоторое гильбертово пространство $\mathcal{H}$ (пространство признаков), в котором данные становятся (почти) линейно разделимыми, и затем применить линейный алгоритм — персептрон, SVM, логистическую регрессию. Ключевая проблема заключается в том, что для многих полезных отображений размерность $\mathcal{H}$ становится огромной или бесконечной. Например, пространство всех полиномиальных признаков степени не выше $d$ для $D$-мерных данных имеет размерность $\binom{D+d}{d} = O(D^d)$, что делает явное построение векторов $\phi(\mathbf{x})$ и операции с ними крайне неэффективными. Для ещё более выразительных отображений, таких как соответствующее RBF-ядру, $\mathcal{H}$ и вовсе бесконечномерно, и явное представление признаков отсутствует в принципе.

---

#### 1.2. Ядро как замена скалярного произведения

Заметим, что многие линейные алгоритмы — как при обучении, так и при предсказании — зависят от данных *только* через скалярные произведения $\langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle_{\mathcal{H}}$. Это ключевое свойство становится очевидным, если переписать алгоритм в двойственной форме. Рассмотрим, например, гребневую регрессию в пространстве признаков:

$$
\mathcal{L}(\mathbf{w}) = \sum_{i=1}^N \bigl(y_i - \langle \mathbf{w}, \phi(\mathbf{x}_i) \rangle_{\mathcal{H}}\bigr)^2 + \lambda \|\mathbf{w}\|_{\mathcal{H}}^2, \qquad \lambda > 0.
$$

Согласно теореме о представителе (которую мы рассмотрим позже), оптимальный вектор весов имеет вид $\mathbf{w} = \sum_{j=1}^N \alpha_j \phi(\mathbf{x}_j)$. Подставляя это представление в функционал, получаем двойственную задачу:

$$
\min_{\boldsymbol{\alpha} \in \mathbb{R}^N} \bigl\| \mathbf{y} - \mathbf{G}\boldsymbol{\alpha} \bigr\|_2^2 + \lambda \boldsymbol{\alpha}^\top \mathbf{G} \boldsymbol{\alpha},
$$

где матрица $\mathbf{G}$ (матрица Грама) составлена из попарных скалярных произведений: $\mathbf{G}_{ij} = \langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle_{\mathcal{H}}$. Предсказание для нового объекта $\mathbf{x}$ также записывается в терминах тех же скалярных произведений:

$$
\hat{y}(\mathbf{x}) = \langle \mathbf{w}, \phi(\mathbf{x}) \rangle_{\mathcal{H}} = \sum_{j=1}^N \alpha_j \langle \phi(\mathbf{x}_j), \phi(\mathbf{x}) \rangle_{\mathcal{H}}.
$$

Таким образом, если нам известна функция двух переменных

$$
K(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}},
$$

называемая **ядром**, мы можем заменить каждое скалярное произведение на вычисление $K(\cdot, \cdot)$, *ни разу не вычисляя само отображение $\phi$*. В этом и состоит **ядровой трюк**: алгоритм переписывается исключительно в терминах ядровой функции, и его вычислительная сложность определяется только числом обучающих примеров и стоимостью вычисления $K$, но не размерностью $\mathcal{H}$ — она может быть сколь угодно большой или даже бесконечной.

**Определение 1 (Ядровая функция).** *Ядром* называется симметричная функция $K : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$, для которой существует гильбертово пространство $\mathcal{H}$ и отображение $\phi : \mathcal{X} \to \mathcal{H}$ такие, что

$$
K(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}}, \quad \forall \mathbf{x}, \mathbf{x}' \in \mathcal{X}.
$$

Таким образом, ядро служит мерой сходства объектов в скрытом пространстве признаков. Его выбор определяет, какую геометрию мы навязываем данным, и часто диктуется либо пониманием природы задачи, либо универсальными рекомендациями (например, RBF-ядро для гладких решающих границ).

---

#### 1.3. Примеры простых ядер

Приведём наиболее распространённые ядровые функции для векторных данных $\mathbf{x}, \mathbf{x}' \in \mathbb{R}^D$.

* **Линейное ядро**  
  $K_{\text{lin}}(\mathbf{x}, \mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$.  
  Соответствует тривиальному отображению $\phi(\mathbf{x}) = \mathbf{x}$ и пространству $\mathcal{H} = \mathbb{R}^D$. Не даёт выигрыша в размерности, но позволяет унифицировать программные реализации.

* **Полиномиальное ядро степени $d$**  
  $K_{\text{poly}}(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^\top \mathbf{x}' + r)^d, \quad \gamma > 0, \; r \ge 0.$  
  Неявно порождает пространство всех мономов от компонент $\mathbf{x}$ степени не выше $d$. Явное отображение $\phi$ содержит $\binom{D+d}{d}$ координат, однако само ядро вычисляется за $O(D)$ — линейно по размерности исходного пространства. Классический пример — разделение XOR: четыре точки в $\mathbb{R}^2$ не разделимы прямой, но становятся разделимы после отображения $\phi(x_1,x_2) = (x_1, x_2, x_1 x_2)$ и применения линейного классификатора.

* **RBF-ядро (гауссово)**  
  $K_{\text{RBF}}(\mathbf{x}, \mathbf{x}') = \exp\bigl(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2\bigr), \quad \gamma > 0.$  
  Это ядро соответствует бесконечномерному гильбертову пространству. Оно является универсальным аппроксиматором: линейная модель в этом пространстве может сколь угодно точно приблизить любую непрерывную функцию на компакте.

* **Сигмоидальное ядро**  
  $K_{\text{sigmoid}}(\mathbf{x}, \mathbf{x}') = \tanh(\gamma \mathbf{x}^\top \mathbf{x}' + r).$  
  Исторически мотивировано нейронными сетями. Однако оно **не всегда** удовлетворяет условию положительной определённости (см. ниже), поэтому его использование в строгих ядровых методах ограничено.

---

#### 1.4. Зачем нужны ядра, если можно просто добавить полиномиальные признаки?

На первый взгляд, для получения нелинейной модели достаточно вручную расширить признаковое пространство — например, добавить все квадратичные взаимодействия. Однако это немедленно ведёт к «проклятию размерности»: число признаков растёт экспоненциально с ростом степени полинома. Так, для $D = 100$ и $d = 3$ число мономов $\binom{103}{3} \approx 176\,851$; для $d = 10$ оно становится астрономическим. Явное построение, хранение и умножение таких векторов требуют колоссальных ресурсов. Полиномиальное же ядро вычисляется за $O(D)$ независимо от $d$.

Но главное преимущество ядер — возможность работать с **бесконечномерными** пространствами, такими как порождаемое RBF-ядром. Его явное представление невозможно, однако все операции через ядро остаются корректными и эффективными. Именно эта способность неявно оперировать в бесконечномерных пространствах, контролируя сложность модели с помощью регуляризации в $\mathcal{H}$, делает ядровые методы столь гибкими.

---

#### 1.5. Углублённый взгляд: явный вид $\phi$ для полиномиального ядра и геометрия RBF-пространства

> **Для читателя с математической подготовкой.** В этом блоке мы детально разберём, как устроено отображение $\phi$ для полиномиального ядра, и исследуем геометрические свойства пространства признаков RBF.

**Явное отображение для полиномиального ядра.** Рассмотрим полиномиальное ядро второй степени с параметрами $\gamma = 1$, $r = 1$:
$$
K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}' + 1)^2.
$$
Разложим квадрат суммы:
$$
(\mathbf{x}^\top \mathbf{x}' + 1)^2 = \left(\sum_{i=1}^D x_i x_i' + 1\right)^2 = \sum_{i=1}^D \sum_{j=1}^D x_i x_j \, x_i' x_j' + 2\sum_{i=1}^D x_i x_i' + 1.
$$
Чтобы представить это выражение как стандартное скалярное произведение $\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle$ в евклидовом пространстве, сгруппируем слагаемые и введём нормирующие множители. Заметим, что каждое парное произведение $x_i x_j$ с $i \neq j$ появляется дважды (как $x_i x_j$ и $x_j x_i$). Поэтому
$$
\phi(\mathbf{x}) = \bigl(x_1^2, \dots, x_D^2, \sqrt{2}x_1 x_2, \dots, \sqrt{2}x_i x_j \;(i<j), \dots, \sqrt{2}x_1, \dots, \sqrt{2}x_D, 1\bigr)^\top.
$$
Число координат в $\phi$ равно числу различных мономов степени $\le 2$ от $D$ переменных, то есть $\binom{D+2}{2} = \frac{(D+2)(D+1)}{2} = O(D^2)$. При этом вычисление ядра $K(\mathbf{x}, \mathbf{x}')$ требует лишь $O(D)$ операций. Для произвольной степени $d$ размерность пространства составит $\binom{D+d}{d} = O(D^d)$, тогда как ядро по-прежнему вычисляется за $O(D)$.

**Геометрия RBF-ядра.** Для RBF-ядра $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$ выполним аналогичный анализ. Прежде всего, $K(\mathbf{x}, \mathbf{x}) = e^0 = 1$, следовательно, квадрат нормы образа любой точки равен единице:
$$
\|\phi(\mathbf{x})\|_{\mathcal{H}}^2 = \langle \phi(\mathbf{x}), \phi(\mathbf{x}) \rangle_{\mathcal{H}} = K(\mathbf{x}, \mathbf{x}) = 1.
$$
Это означает, что **все объекты отображаются на поверхность единичной сферы** в бесконечномерном гильбертовом пространстве $\mathcal{H}$.

Квадрат расстояния между образами двух точек выражается через ядро:
$$
\|\phi(\mathbf{x}) - \phi(\mathbf{x}')\|_{\mathcal{H}}^2 = K(\mathbf{x},\mathbf{x}) + K(\mathbf{x}',\mathbf{x}') - 2K(\mathbf{x},\mathbf{x}') = 2 - 2\exp\bigl(-\gamma\|\mathbf{x} - \mathbf{x}'\|^2\bigr).
$$

Проанализируем это выражение:

* **Локальное поведение.** Если точки $\mathbf{x}$ и $\mathbf{x}'$ близки в исходном пространстве ($\|\mathbf{x} - \mathbf{x}'\|^2 \ll 1/\gamma$), то, используя разложение $e^{-t} \approx 1 - t$, получаем
  $$
  \|\phi(\mathbf{x}) - \phi(\mathbf{x}')\|_{\mathcal{H}}^2 \approx 2 - 2\bigl(1 - \gamma\|\mathbf{x} - \mathbf{x}'\|^2\bigr) = 2\gamma \|\mathbf{x} - \mathbf{x}'\|^2.
  $$
  Таким образом, в малой окрестности индуцированное расстояние в $\mathcal{H}$ пропорционально евклидову расстоянию в $\mathcal{X}$ — пространство признаков локально ведёт себя как обычное евклидово.

* **Глобальное поведение (насыщение).** Если точки далеки ($\|\mathbf{x} - \mathbf{x}'\|^2 \gg 1/\gamma$), то экспонента стремится к нулю, и расстояние в $\mathcal{H}$ стремится к константе:
  $$
  \|\phi(\mathbf{x}) - \phi(\mathbf{x}')\|_{\mathcal{H}}^2 \longrightarrow 2.
  $$
  Поскольку оба образа лежат на сфере радиуса 1, максимальное возможное расстояние между ними равно $\sqrt{2}$ (диаметр сферы). Следовательно, далёкие друг от друга в исходном пространстве точки становятся почти ортогональными: $\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}} \approx 0$.

Эта нелинейная метрика объясняет эффективность RBF: локальная структура данных сохраняется, а глобально все далёкие точки «разбегаются» на противоположные стороны сферы, что облегчает построение линейной разделяющей гиперплоскости в $\mathcal{H}$.

**Необходимое условие: положительная определённость.** Чтобы функция $K$ действительно была ядром (т.е. существовало гильбертово пространство $\mathcal{H}$ и отображение $\phi$ с $K(\mathbf{x},\mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}}$), она должна удовлетворять условию положительной определённости: для любого конечного набора точек $\mathbf{x}_1, \dots, \mathbf{x}_n$ и любых коэффициентов $c_1, \dots, c_n \in \mathbb{R}$ выполняется
$$
\sum_{i=1}^n \sum_{j=1}^n c_i c_j K(\mathbf{x}_i, \mathbf{x}_j) \ge 0.
$$
Это свойство гарантирует, что матрица Грама $\mathbf{G}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$ неотрицательно определена. Линейное, полиномиальное и RBF-ядра этому условию удовлетворяют. Сигмоидальное ядро — нет. В качестве контрпримера можно взять три точки, например, $x_1 = -1, x_2 = 0, x_3 = 1$ и параметры $\gamma = 1, r = 0$. Тогда матрица Грама будет иметь элементы $K_{ij} = \tanh(x_i x_j)$, и её собственные значения не все неотрицательны (одно из них приблизительно равно $-0.23$). Таким образом, сигмоидальное ядро не определяет допустимое гильбертово пространство, и его применение в SVM или других строгих ядровых методах может приводить к потере выпуклости задачи и численной неустойчивости.

---

#### Вопросы для самопроверки

**Для начинающих:**

1.  Что такое ядровой трюк и какую вычислительную проблему он решает? Сформулируйте основную идею своими словами.
2.  Зачем отображать данные в пространство большей размерности? Приведите пример двух классов в $\mathbb{R}^2$, которые неразделимы прямой, но становятся разделимы плоскостью в $\mathbb{R}^3$ после отображения $\phi(x_1, x_2) = (x_1, x_2, x_1^2 + x_2^2)$.
3.  Почему во многих случаях нельзя или неэффективно явно строить отображение $\phi$? Оцените число признаков для $D = 50$, $d = 3$ в полиномиальном отображении и сравните с вычислительной сложностью вычисления ядра $K = (\mathbf{x}^\top \mathbf{x}' + 1)^3$.
4.  Приведите пример набора данных, который линейно неразделим, но становится разделимым после добавления квадратов признаков.
5.  Опишите RBF-ядро: его формулу и основное свойство. Как меняется его значение при увеличении расстояния между $\mathbf{x}$ и $\mathbf{x}'$?

**Для профессионалов:**

1.  Докажите, что полиномиальное ядро степени $2$ для $D$-мерных данных соответствует отображению в пространство размерности $O(D^2)$. Выпишите явный вид $\phi$ для $D = 2$ с точными нормирующими коэффициентами.
2.  Используя разложение экспоненты в ряд Тейлора, покажите, что RBF-ядро является скалярным произведением в бесконечномерном пространстве. Какие координаты имеет $\phi(\mathbf{x})$ в этом представлении?
3.  Приведите пример реальной задачи (например, из компьютерного зрения или биоинформатики), где линейное ядро не способно уловить структуру данных, и объясните, почему RBF-ядро справляется лучше. Какая геометрическая особенность данных ответственна за это?
4.  Сигмоидальное ядро $\tanh(\gamma \mathbf{x}^\top \mathbf{x}' + r)$ не всегда является положительно определённым. Приведите конкретный контрпример: выберите три точки в $\mathbb{R}$ и подберите параметры $\gamma, r$ так, чтобы матрица Грама имела отрицательное собственное значение.

Вот итоговый, максимально полный вариант раздела K2, объединяющий ваши наработки и мои детали.

---

### 2. Положительно определённые ядра и теорема Мерсера

В предыдущем разделе мы ввели понятие ядра как функции, задающей скалярное произведение в спрямляющем пространстве: $K(\mathbf{x},\mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}')\rangle_{\mathcal{H}}$. Однако на практике мы хотим строить ядра непосредственно, не конструируя явно $\phi$ и $\mathcal{H}$. Возникает фундаментальный вопрос: каким условиям должна удовлетворять функция $K$, чтобы она действительно была ядром — то есть чтобы существовали гильбертово пространство $\mathcal{H}$ и отображение $\phi$, порождающие $K$ как скалярное произведение? Ответ на него дают понятие положительной определённости и теорема Мерсера, составляющие математический фундамент ядровых методов.

#### 2.1. Определение положительно определённого ядра

Пусть $\mathcal{X}$ — произвольное множество. Функция $K : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ называется **положительно определённым (ПО) ядром**, если она симметрична, т.е.

$$
K(\mathbf{x}, \mathbf{y}) = K(\mathbf{y}, \mathbf{x}) \quad \forall \mathbf{x}, \mathbf{y} \in \mathcal{X},
$$

и для любого конечного набора точек $\mathbf{x}_1,\dots,\mathbf{x}_n \in \mathcal{X}$ и любых вещественных чисел $c_1,\dots,c_n \in \mathbb{R}$ выполняется неравенство

$$
\sum_{i=1}^n \sum_{j=1}^n c_i c_j K(\mathbf{x}_i, \mathbf{x}_j) \ge 0. \tag{2.1}
$$

Геометрический смысл этого условия прозрачен: квадратичная форма, построенная по матрице Грама $\mathbf{G}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$, должна быть неотрицательной. Другими словами, для любого конечного набора точек матрица $\mathbf{G}$ должна быть неотрицательно определённой. Именно это свойство, согласно теореме Мура–Ароншайна, гарантирует, что функция $K$ может быть представлена как скалярное произведение в некотором гильбертовом пространстве — воспроизводящем ядерном гильбертовом пространстве (RKHS), однозначно определяемом ядром.

**Пример.** Линейное ядро $K(\mathbf{x},\mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$ является ПО, поскольку для любых $c_i$ имеем
$$
\sum_{i,j} c_i c_j \mathbf{x}_i^\top \mathbf{x}_j = \Bigl\|\sum_i c_i \mathbf{x}_i\Bigr\|^2 \ge 0.
$$

**Контрпример.** Функция $K(x,y) = \tanh(\gamma x y + r)$ при определённых параметрах не является ПО, так как соответствующая матрица Грама для трёх точек может иметь отрицательное собственное значение (см. упражнения).

Положительная определённость — это не только формальный критерий, но и практический инструмент: имея обучающую выборку, мы можем проверить, что матрица $\mathbf{G}$ положительно полуопределена, однако для гарантии на всём пространстве $\mathcal{X}$ необходимы более глубокие характеризации.

#### 2.2. Теорема Мерсера

Для непрерывных ядер на компактных подмножествах $\mathbb{R}^D$ условие ПО эквивалентно существованию спектрального разложения с неотрицательными коэффициентами. Этот факт составляет содержание классической теоремы Мерсера.

**Теорема 2.1 (Мерсер).** *Пусть $\mathcal{X} \subset \mathbb{R}^D$ — компактное множество, $\mu$ — конечная строго положительная мера на $\mathcal{X}$, и $K : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ — непрерывная симметричная положительно определённая функция. Тогда существуют неотрицательные числа $\lambda_1 \ge \lambda_2 \ge \dots \ge 0$ и система непрерывных функций $\{\psi_k\}_{k=1}^\infty \subset L_2(\mathcal{X}, \mu)$, образующих ортонормированный базис в $L_2(\mathcal{X}, \mu)$, такие что*

$$
K(\mathbf{x}, \mathbf{x}') = \sum_{k=1}^\infty \lambda_k \psi_k(\mathbf{x}) \psi_k(\mathbf{x}'), \tag{2.2}
$$

*причём ряд сходится абсолютно и равномерно на $\mathcal{X} \times \mathcal{X}$.*

Функции $\psi_k$ являются собственными функциями интегрального оператора $T_K$, ассоциированного с ядром:

$$
(T_K f)(\mathbf{x}) = \int_{\mathcal{X}} K(\mathbf{x}, \mathbf{y}) f(\mathbf{y}) \, d\mu(\mathbf{y}),
$$

причём $T_K \psi_k = \lambda_k \psi_k$. Теорема Мерсера утверждает, что ядро раскладывается по этим собственным функциям, а собственные числа неотрицательны и стремятся к нулю: $\lambda_k \to 0$ при $k \to \infty$. Интуитивно, $\lambda_k$ показывают «вес» соответствующей базисной функции в ядре. Если лишь конечное число $\lambda_k$ отличны от нуля, размерность спрямляющего пространства конечна; если же все $\lambda_k > 0$, пространство бесконечномерно. Скорость убывания $\lambda_k$ характеризует гладкость и сложность ядра: быстрое убывание означает эффективную конечномерность и сильную регуляризацию.

#### 2.3. Построение спрямляющего пространства через теорему Мерсера

Имея разложение (2.2), мы можем явно построить гильбертово пространство $\mathcal{H}$ и отображение $\phi$, реализующие ядро как скалярное произведение. Определим

$$
\phi(\mathbf{x}) = \bigl( \sqrt{\lambda_1}\,\psi_1(\mathbf{x}),\; \sqrt{\lambda_2}\,\psi_2(\mathbf{x}),\; \dots \bigr)^\top.
$$

Тогда для любых $\mathbf{x}, \mathbf{x}' \in \mathcal{X}$ имеем

$$
\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\ell_2} = \sum_{k=1}^\infty \lambda_k \psi_k(\mathbf{x}) \psi_k(\mathbf{x}') = K(\mathbf{x}, \mathbf{x}').
$$

Таким образом, $\phi$ отображает исходное пространство в пространство квадратично суммируемых последовательностей $\ell_2$ (являющееся гильбертовым), и $K$ в точности равно скалярному произведению образов. Теорема Мерсера даёт конструктивный способ построения $\phi$, но на практике мы не вычисляем $\phi$ явно: для работы алгоритмов достаточно знать само ядро.

Заметим, что размерность $\mathcal{H}$ равна числу ненулевых $\lambda_k$. Для ядер типа RBF все $\lambda_k > 0$, и пространство оказывается бесконечномерным — однако ядровой трюк позволяет работать с ним, не прибегая к бесконечным вычислениям.

#### 2.4. Проверка положительной определённости для стандартных ядер

Покажем, что три основных ядра удовлетворяют условию ПО.

**Линейное ядро** $K_{\text{lin}}(\mathbf{x}, \mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$.  
Для любого набора точек $\mathbf{x}_1, \dots, \mathbf{x}_n$ и весов $c_i$ имеем
$$
\sum_{i,j} c_i c_j \mathbf{x}_i^\top \mathbf{x}_j = \Bigl\| \sum_i c_i \mathbf{x}_i \Bigr\|^2 \ge 0,
$$
так что ядро положительно определено.

**Полиномиальное ядро** $K_{\text{poly}}(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^\top \mathbf{x}' + r)^d$, $\gamma > 0, r \ge 0$.  
Доказательство ПО опирается на свойства замкнутости класса ПО ядер, подробно рассмотренные в углублённом взгляде (раздел 2.5). Вкратце:

- Линейное ядро $\gamma \mathbf{x}^\top \mathbf{x}'$ ПО.
- Константное ядро $K_r(\mathbf{x}, \mathbf{x}') = r$ ПО, так как его матрица Грама состоит из одинаковых элементов $r$ и является неотрицательно определённой при $r \ge 0$: квадратичная форма $\sum_{i,j} c_i c_j r = r (\sum_i c_i)^2 \ge 0$.
- Сумма ПО ядер ПО, следовательно, $K_1(\mathbf{x}, \mathbf{x}') = \gamma \mathbf{x}^\top \mathbf{x}' + r$ — ПО ядро.
- Произведение ПО ядер ПО (теорема Шура), а возведение в степень $d$ есть $d$-кратное произведение ядра на себя. Поэтому $K_{\text{poly}} = (K_1)^d$ — ПО ядро.

**RBF-ядро** $K_{\text{RBF}}(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$, $\gamma > 0$.  
Это стационарное ядро: оно зависит только от разности $\mathbf{h} = \mathbf{x} - \mathbf{x}'$. Для таких ядер имеется мощный критерий — теорема Бохнера, которую мы рассмотрим в углублённом взгляде ниже.

**Сигмоидальное ядро** $K_{\text{sigmoid}}(\mathbf{x}, \mathbf{x}') = \tanh(\gamma \mathbf{x}^\top \mathbf{x}' + r)$.  
Как уже отмечалось, эта функция не является ПО при всех параметрах (матрица Грама для трёх точек может иметь отрицательное собственное значение), поэтому её использование в методах, требующих ПО ядра, ограничено.

#### 2.5. Углублённый взгляд: теорема Бохнера и свойства замкнутости класса ПО ядер

> **Для читателя с математической подготовкой.** Этот блок даёт два фундаментальных инструмента: полную характеризацию стационарных ядер через преобразование Фурье (теорема Бохнера) и операции, сохраняющие положительную определённость.

**Теорема Бохнера для стационарных ядер.**  
Многие важные ядра, включая RBF, являются стационарными: $K(\mathbf{x}, \mathbf{x}') = k(\mathbf{x} - \mathbf{x}')$, где $k : \mathbb{R}^D \to \mathbb{R}$ — непрерывная функция одного векторного аргумента. Для таких ядер положительная определённость полностью характеризуется через преобразование Фурье.

**Теорема 2.2 (Бохнер).** *Непрерывная функция $k : \mathbb{R}^D \to \mathbb{C}$ является положительно определённой (т.е. $K(\mathbf{x},\mathbf{x}') = k(\mathbf{x}-\mathbf{x}')$ — ПО ядро) тогда и только тогда, когда она представима в виде преобразования Фурье–Стилтьеса конечной неотрицательной борелевской меры $\mu$ на $\mathbb{R}^D$:*

$$
k(\mathbf{h}) = \int_{\mathbb{R}^D} e^{-i \mathbf{h}^\top \boldsymbol{\omega}} \, d\mu(\boldsymbol{\omega}), \qquad \forall \mathbf{h} \in \mathbb{R}^D.
$$

Если мера $\mu$ абсолютно непрерывна относительно меры Лебега, то существует спектральная плотность $p(\boldsymbol{\omega}) \ge 0$, и $k$ является обратным преобразованием Фурье от $p$:

$$
k(\mathbf{h}) = \int_{\mathbb{R}^D} e^{-i \mathbf{h}^\top \boldsymbol{\omega}} p(\boldsymbol{\omega}) \, d\boldsymbol{\omega}.
$$

Применим теорему к RBF-ядру. Для $k(\mathbf{h}) = \exp(-\gamma \|\mathbf{h}\|^2)$ прямое преобразование Фурье (в смысле обычного интегрирования) даёт

$$
\hat{k}(\boldsymbol{\omega}) = \int_{\mathbb{R}^D} e^{-i \mathbf{h}^\top \boldsymbol{\omega}} e^{-\gamma \|\mathbf{h}\|^2} d\mathbf{h} = \left(\frac{\pi}{\gamma}\right)^{D/2} \exp\!\Bigl(-\frac{\|\boldsymbol{\omega}\|^2}{4\gamma}\Bigr) > 0, \quad \forall \boldsymbol{\omega} \in \mathbb{R}^D.
$$

Таким образом, спектральная плотность $p(\boldsymbol{\omega}) = (2\pi)^{-D} \hat{k}(\boldsymbol{\omega})$ строго положительна. По теореме Бохнера $k$ положительно определена, а значит, $K_{\text{RBF}}$ — допустимое ядро. Более того, из строгой положительности плотности следует, что все собственные числа $\lambda_k$ в разложении Мерсера строго положительны — пространство признаков бесконечномерно.

Это доказательство не только подтверждает корректность RBF-ядра, но и раскрывает его вероятностную природу: гауссовская спектральная плотность отражает бесконечную гладкость и локальность ядра.

**Свойства замкнутости класса ПО ядер.**  
ПО ядра образуют выпуклый конус, замкнутый относительно нескольких операций:

1. *Сумма.* Если $K_1$ и $K_2$ — ПО ядра, то $K_1 + K_2$ — ПО ядро. Это непосредственно следует из определения (2.1): для любой матрицы Грама $\mathbf{G} = \mathbf{G}_1 + \mathbf{G}_2$ квадратичная форма неотрицательна как сумма неотрицательных.

2. *Произведение.* Если $K_1$ и $K_2$ — ПО ядра, то их поточечное произведение $K(\mathbf{x}, \mathbf{x}') = K_1(\mathbf{x}, \mathbf{x}') K_2(\mathbf{x}, \mathbf{x}')$ — также ПО ядро. Доказательство опирается на теорему Шура: поэлементное (адамарово) произведение двух неотрицательно определённых матриц является неотрицательно определённым. Для любых конечных наборов точек матрицы $\mathbf{G}_1$ и $\mathbf{G}_2$ неотрицательно определены; их поэлементное произведение $\mathbf{G} = \mathbf{G}_1 \odot \mathbf{G}_2$ неотрицательно определено, что равносильно выполнению (2.1) для $K$.

3. *Многочлены с неотрицательными коэффициентами.* Если $K$ — ПО ядро и $p(z) = \sum_{m=0}^d a_m z^m$, где $a_m \ge 0$, то $p(K(\mathbf{x}, \mathbf{x}'))$ — ПО ядро. Это прямое следствие свойств 1 и 2 и того факта, что константное ядро $K \equiv 1$ ПО.

4. *Предельный переход.* Если последовательность ПО ядер $\{K_n\}$ сходится поточечно к функции $K$, то $K$ также является ПО ядром. Действительно, неравенство (2.1) сохраняется при предельном переходе.

Эти свойства позволяют конструировать новые ядра из элементарных и доказывать ПО широкого класса функций, в том числе полиномиальных и ядер на основе экспоненты (через разложение в ряд).

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что означает утверждение «ядро положительно определено»? Запишите формальное определение своими словами и приведите пример ПО ядра и пример функции, не являющейся ПО ядром.
2. Какую роль играет теорема Мерсера в ядровых методах? Почему она важна для связи ядер и гильбертовых пространств?
3. Объясните, как из спектрального разложения ядра (теорема Мерсера) строится отображение $\phi$ в спрямляющее пространство. Какова размерность этого пространства?
4. Опишите, как проверить, что RBF-ядро является допустимым ядром, используя теорему Бохнера или разложение в ряд.
5. Что такое компактное множество? Приведите пример множества в $\mathbb{R}^D$, которое компактно, и пример, которое некомпактно. Почему условие компактности важно в теореме Мерсера?

**Для профессионалов:**

1. Докажите, что сумма двух ПО ядер является ПО ядром. Покажите, что класс ПО ядер образует выпуклый конус.
2. Докажите, что произведение двух ПО ядер также является ПО ядром. (Указание: воспользуйтесь теоремой Шура о поэлементном произведении неотрицательно определённых матриц и конечномерной аппроксимацией.)
3. Приведите конкретный численный пример (с тремя точками в $\mathbb{R}$), когда функция $\tanh(\gamma x y + r)$ даёт матрицу Грама с отрицательным собственным значением при некоторых $\gamma, r$. Укажите параметры и вычислите собственные значения.
4. Выпишите разложение Мерсера для полиномиального ядра $K(\mathbf{x},\mathbf{x}') = (\mathbf{x}^\top \mathbf{x}')^2$ в $\mathbb{R}^2$ с равномерной мерой на единичном круге или квадрате. Найдите явно собственные функции и собственные числа, опираясь на представление через мономы.
5. Объясните, как для конечной выборки проверить, является ли данная функция $K$ ПО ядром. Какие ограничения у такой конечномерной проверки? Почему для полной гарантии необходимо проверять условие для всех возможных конечных наборов точек?

Ниже представлена итоговая, гибридная и наиболее полная версия раздела K3, объединяющая ваши наработки и мои детали в единое связное повествование.

---

### 3. Воспроизводящее ядерное гильбертово пространство (RKHS)

Мы установили, что положительно определённое ядро задаёт скалярное произведение в некотором гильбертовом пространстве $\mathcal{H}$ через отображение $\phi$: $K(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}}$. Однако в машинном обучении нас интересуют не просто абстрактные векторы, а **функции** — предсказывающие модели. Оказывается, каждому положительно определённому ядру можно сопоставить единственное гильбертово пространство, состоящее из функций на $\mathcal{X}$, в котором все операции — норма, скалярное произведение, оценка функции в точке — выражаются через ядро. Это пространство называется **воспроизводящим ядерным гильбертовым пространством** (Reproducing Kernel Hilbert Space, RKHS). Оно является «родным домом» для решений ядровых методов, а теорема о представителе объясняет, почему оптимальные функции всегда имеют вид линейной комбинации ядер, привязанных к обучающим точкам.

---

#### 3.1. Определение RKHS

Пусть $\mathcal{X}$ — произвольное множество, и $\mathcal{H}$ — гильбертово пространство вещественных функций $f : \mathcal{X} \to \mathbb{R}$ (сложение и умножение на скаляр определены поточечно). Пространство $\mathcal{H}$ называется **воспроизводящим ядерным гильбертовым пространством (RKHS)**, если для любого $\mathbf{x} \in \mathcal{X}$ линейный функционал *оценки* (evaluation functional)

$$
L_{\mathbf{x}} : \mathcal{H} \to \mathbb{R}, \qquad L_{\mathbf{x}}(f) = f(\mathbf{x}),
$$

является непрерывным (ограниченным). Иными словами, для каждого $\mathbf{x}$ существует константа $C_{\mathbf{x}} \ge 0$ такая, что $|f(\mathbf{x})| \le C_{\mathbf{x}} \|f\|_{\mathcal{H}}$ для всех $f \in \mathcal{H}$.

Требование непрерывности функционала оценки — ключевое. Именно оно отделяет RKHS от «обычных» гильбертовых пространств функций, таких как $L_2$, где значение функции в точке, вообще говоря, не определено. В RKHS сходимость по норме влечёт поточечную сходимость: если $\|f_n - f\|_{\mathcal{H}} \to 0$, то для любого $\mathbf{x}$
$$
|f_n(\mathbf{x}) - f(\mathbf{x})| = |L_{\mathbf{x}}(f_n - f)| \le C_{\mathbf{x}} \|f_n - f\|_{\mathcal{H}} \to 0.
$$

По теореме Рисса о представлении линейного непрерывного функционала в гильбертовом пространстве, для каждого $\mathbf{x} \in \mathcal{X}$ существует единственный элемент $K_{\mathbf{x}} \in \mathcal{H}$ такой, что

$$
f(\mathbf{x}) = \langle f, K_{\mathbf{x}} \rangle_{\mathcal{H}}, \qquad \forall f \in \mathcal{H}. \tag{3.1}
$$

Функция $K_{\mathbf{x}} = K(\cdot, \mathbf{x})$ называется **воспроизводящим элементом** для точки $\mathbf{x}$, а равенство (3.1) — **воспроизводящим свойством**: значение любой функции $f \in \mathcal{H}$ в точке $\mathbf{x}$ восстанавливается как скалярное произведение с $K_{\mathbf{x}}$. Геометрически это означает, что оценка функции в точке — это просто проекция на фиксированный вектор. В конечномерном евклидовом пространстве аналогом служит равенство $w_i = \langle w, e_i \rangle$, где $e_i$ — базисный вектор.

---

#### 3.2. Воспроизводящее ядро

Определим функцию двух переменных

$$
K(\mathbf{x}, \mathbf{y}) = \langle K_{\mathbf{x}}, K_{\mathbf{y}} \rangle_{\mathcal{H}}, \qquad \mathbf{x}, \mathbf{y} \in \mathcal{X}. \tag{3.2}
$$

Эта функция называется **воспроизводящим ядром** пространства $\mathcal{H}$. Из определения немедленно вытекают следующие свойства:

* **Симметричность:** $K(\mathbf{x}, \mathbf{y}) = K(\mathbf{y}, \mathbf{x})$ в силу симметричности скалярного произведения.
* **Положительная определённость:** для любого конечного набора точек $\{\mathbf{x}_i\}_{i=1}^n \subset \mathcal{X}$ и любых коэффициентов $c_i \in \mathbb{R}$
  $$
  \sum_{i=1}^n \sum_{j=1}^n c_i c_j K(\mathbf{x}_i, \mathbf{x}_j) = \sum_{i,j} c_i c_j \langle K_{\mathbf{x}_i}, K_{\mathbf{x}_j} \rangle_{\mathcal{H}} = \Bigl\| \sum_{i=1}^n c_i K_{\mathbf{x}_i} \Bigr\|_{\mathcal{H}}^2 \ge 0.
  $$
  Таким образом, $K$ является положительно определённым ядром в смысле определения раздела 2.
* **Воспроизводящее свойство для аргументов ядра:** для каждого $\mathbf{y} \in \mathcal{X}$ функция $K(\cdot, \mathbf{y})$ совпадает с воспроизводящим элементом $K_{\mathbf{y}}$, поэтому
  $$
  f(\mathbf{y}) = \langle f, K(\cdot, \mathbf{y}) \rangle_{\mathcal{H}}, \qquad \forall f \in \mathcal{H}. \tag{3.3}
  $$
  В частности, $K(\mathbf{x}, \mathbf{y}) = \langle K(\cdot, \mathbf{x}), K(\cdot, \mathbf{y}) \rangle_{\mathcal{H}}$.

Следовательно, **каждое RKHS порождает положительно определённое ядро**. Верно и обратное: по любому положительно определённому ядру $K$ можно построить единственное RKHS, для которого $K$ является воспроизводящим. Конструкция такова: на линейной оболочке функций $\{K(\cdot, \mathbf{x}) \mid \mathbf{x} \in \mathcal{X}\}$ вводится скалярное произведение

$$
\Bigl\langle \sum_i \alpha_i K(\cdot, \mathbf{x}_i), \sum_j \beta_j K(\cdot, \mathbf{y}_j) \Bigr\rangle_{\mathcal{H}} = \sum_{i,j} \alpha_i \beta_j K(\mathbf{x}_i, \mathbf{y}_j).
$$

Пополнение этой линейной оболочки относительно порождённой нормы даёт гильбертово пространство $\mathcal{H}$, в котором воспроизводящее свойство (3.1) выполнено по определению. Это — знаменитая **теорема Мура–Ароншайна**, устанавливающая взаимно-однозначное соответствие между положительно определёнными ядрами и RKHS.

---

#### 3.3. Свойства RKHS

1. **Неравенство Коши–Буняковского и поточечная оценка.** Из воспроизводящего свойства и неравенства Коши–Буняковского немедленно следует
   $$
   |f(\mathbf{x})| = |\langle f, K_{\mathbf{x}} \rangle_{\mathcal{H}}| \le \|f\|_{\mathcal{H}} \|K_{\mathbf{x}}\|_{\mathcal{H}} = \|f\|_{\mathcal{H}} \sqrt{K(\mathbf{x}, \mathbf{x})}. \tag{3.4}
   $$
   Таким образом, функционал оценки ограничен с константой $C_{\mathbf{x}} = \sqrt{K(\mathbf{x}, \mathbf{x})}$. Это неравенство показывает, что норма в RKHS гораздо сильнее, чем, скажем, $L_2$-норма: сходимость по RKHS-норме влечёт равномерную поточечную сходимость на любом множестве, где $K(\mathbf{x}, \mathbf{x})$ ограничено.

2. **Связь со спрямляющим пространством.** Если ядро представлено как $K(\mathbf{x}, \mathbf{y}) = \langle \phi(\mathbf{x}), \phi(\mathbf{y}) \rangle_{\mathcal{H}_0}$ для некоторого отображения $\phi : \mathcal{X} \to \mathcal{H}_0$, то RKHS $\mathcal{H}$ изоморфно замыканию линейной оболочки $\{\phi(\mathbf{x}) \mid \mathbf{x} \in \mathcal{X}\}$ в $\mathcal{H}_0$. Изоморфизм задаётся отождествлением $f(\mathbf{x}) = \langle \mathbf{w}, \phi(\mathbf{x}) \rangle_{\mathcal{H}_0}$ с элементом $\mathbf{w}$. В этом смысле функции из RKHS — это в точности линейные функционалы в спрямляющем пространстве, и любая такая функция допускает представление $f(\cdot) = \langle \mathbf{w}, \phi(\cdot) \rangle$.

3. **Спектральная структура и гладкость.** Если ядро допускает разложение Мерсера $K(\mathbf{x}, \mathbf{y}) = \sum_{k=1}^\infty \lambda_k \psi_k(\mathbf{x}) \psi_k(\mathbf{y})$, то RKHS $\mathcal{H}$ состоит из функций $f = \sum_{k} a_k \psi_k$, для которых конечна норма
   $$
   \|f\|_{\mathcal{H}}^2 = \sum_{k=1}^\infty \frac{a_k^2}{\lambda_k} < \infty,
   $$
   где $a_k = \int f(\mathbf{x}) \psi_k(\mathbf{x}) d\mu(\mathbf{x})$. Быстрое убывание собственных чисел $\lambda_k$ означает, что коэффициенты $a_k$ должны убывать ещё быстрее; высокочастотные компоненты (большие $k$) сильно подавлены, и функции в $\mathcal{H}$ становятся очень гладкими. Для RBF-ядра $\lambda_k$ убывают экспоненциально, поэтому все функции из соответствующего RKHS являются вещественно-аналитическими. Норма $\|f\|_{\mathcal{H}}$ фактически измеряет взвешенную энергию преобразования Фурье, играя роль штрафа за негладкость.

4. **Плотность в $L_2$.** При условии, что все $\lambda_k > 0$, RKHS плотно в $L_2(\mathcal{X})$ (поскольку содержит все конечные линейные комбинации базисных функций $\psi_k$, которые образуют базис $L_2$). Однако норма $\|\cdot\|_{\mathcal{H}}$ гораздо сильнее $L_2$-нормы: существуют функции, принадлежащие $L_2$, но не принадлежащие RKHS (например, функции с недостаточно быстрым убыванием коэффициентов $a_k$ или разрывные функции). Таким образом, RKHS — это собственное подмножество $L_2$, выделяемое условиями гладкости.

---

#### 3.4. Примеры RKHS

**Линейное ядро.** Рассмотрим $\mathcal{X} = \mathbb{R}^D$ и $K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top \mathbf{y}$. Воспроизводящий элемент: $K_{\mathbf{x}}(\cdot) = \langle \cdot, \mathbf{x} \rangle$. Соответствующее RKHS $\mathcal{H}$ состоит из всех линейных функций $f_{\mathbf{w}}(\mathbf{x}) = \mathbf{w}^\top \mathbf{x}$, где $\mathbf{w} \in \mathbb{R}^D$, с нормой $\|f_{\mathbf{w}}\|_{\mathcal{H}} = \|\mathbf{w}\|_2$. Действительно, $\langle f_{\mathbf{w}}, K_{\mathbf{x}} \rangle_{\mathcal{H}} = \mathbf{w}^\top \mathbf{x} = f_{\mathbf{w}}(\mathbf{x})$. Это пространство изоморфно самому $\mathbb{R}^D$ и содержит лишь линейные функции. Важно: константная функция $f(\mathbf{x}) \equiv 1$ **не** принадлежит этому RKHS, так как её нельзя представить в виде $\mathbf{w}^\top \mathbf{x}$ с конечной евклидовой нормой.

**RBF-ядро.** Для $K_{\text{RBF}}(\mathbf{x}, \mathbf{y}) = \exp(-\gamma \|\mathbf{x} - \mathbf{y}\|^2)$ RKHS $\mathcal{H}$ устроено сложнее. Его можно описать как замыкание линейной оболочки функций $K(\cdot, \mathbf{x}_i)$ по норме, порождённой скалярным произведением через ядро. Все функции из этого пространства бесконечно дифференцируемы, и их норма эквивалентна взвешенной $L_2$-норме преобразования Фурье:
$$
\|f\|_{\mathcal{H}}^2 \propto \int_{\mathbb{R}^D} |\hat{f}(\boldsymbol{\omega})|^2 \exp\!\Bigl(\frac{\|\boldsymbol{\omega}\|^2}{4\gamma}\Bigr) d\boldsymbol{\omega}.
$$
Следовательно, принадлежность к RKHS требует очень быстрого убывания фурье-образа, то есть высокой гладкости. Пространство плотно в $L_2$, но содержит далеко не все $L_2$-функции: любая разрывная функция автоматически не попадает в RKHS.

---

#### 3.5. Углублённый взгляд: теорема о представителе

> **Для читателя с математической подготовкой.** Этот блок содержит центральный результат теории ядровых методов — теорему о представителе. Она объясняет, почему решения задач обучения в RKHS всегда имеют вид конечной линейной комбинации ядровых функций, привязанных к обучающим точкам, и сводит бесконечномерную оптимизацию к конечномерной.

Рассмотрим общую задачу минимизации регуляризованного эмпирического риска:

$$
\min_{f \in \mathcal{H}} \; \mathcal{R}(f) = \sum_{i=1}^n L(y_i, f(\mathbf{x}_i)) + \Omega(\|f\|_{\mathcal{H}}), \tag{3.5}
$$

где $\mathcal{H}$ — RKHS с воспроизводящим ядром $K$, $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ — обучающая выборка, $L$ — произвольная функция потерь, а $\Omega : [0,\infty) \to \mathbb{R}$ — строго монотонно возрастающая функция (например, $\Omega(t) = \lambda t^2$). **Теорема о представителе** утверждает, что существует решение $f^*$ задачи (3.5), представимое в виде

$$
f^*(\mathbf{x}) = \sum_{i=1}^n \alpha_i K(\mathbf{x}, \mathbf{x}_i), \qquad \alpha_i \in \mathbb{R}. \tag{3.6}
$$

**Доказательство.** Рассмотрим подпространство $\mathcal{H}_0 = \operatorname{span}\{K(\cdot, \mathbf{x}_i) : i = 1, \dots, n\} \subset \mathcal{H}$. Поскольку $\mathcal{H}_0$ конечномерно, $\mathcal{H}$ разлагается в ортогональную прямую сумму $\mathcal{H} = \mathcal{H}_0 \oplus \mathcal{H}_\perp$. Для произвольного $f \in \mathcal{H}$ запишем $f = f_0 + f_\perp$, где $f_0 \in \mathcal{H}_0$ и $f_\perp \perp \mathcal{H}_0$. В силу воспроизводящего свойства для любой точки $\mathbf{x}_i$

$$
f(\mathbf{x}_i) = \langle f, K(\cdot, \mathbf{x}_i) \rangle_{\mathcal{H}} = \langle f_0, K(\cdot, \mathbf{x}_i) \rangle_{\mathcal{H}} + \langle f_\perp, K(\cdot, \mathbf{x}_i) \rangle_{\mathcal{H}} = f_0(\mathbf{x}_i) + 0,
$$

так как $K(\cdot, \mathbf{x}_i) \in \mathcal{H}_0$, а $f_\perp \perp \mathcal{H}_0$. Следовательно, значения функции $f$ на обучающих точках определяются исключительно компонентой $f_0$. Эмпирический риск $\sum_i L(y_i, f(\mathbf{x}_i))$ зависит только от $f_0$.

С другой стороны, из ортогональности разложения $\|f\|_{\mathcal{H}}^2 = \|f_0\|_{\mathcal{H}}^2 + \|f_\perp\|_{\mathcal{H}}^2 \ge \|f_0\|_{\mathcal{H}}^2$. Поскольку $\Omega$ строго монотонна, $\Omega(\|f\|_{\mathcal{H}}) \ge \Omega(\|f_0\|_{\mathcal{H}})$, причём равенство достигается только при $f_\perp = 0$. Таким образом, для любого $f$ существует $f_0 \in \mathcal{H}_0$, который имеет те же значения на обучающей выборке, но не больший регуляризатор. Следовательно, оптимальное решение можно искать в $\mathcal{H}_0$, что и даёт представление (3.6). $\square$

**Частный случай: квадратичная потеря и $\ell_2$-регуляризация.** В задаче гребневой регрессии в RKHS
$$
\min_{f \in \mathcal{H}} \left\{ \sum_{i=1}^n (y_i - f(\mathbf{x}_i))^2 + \lambda \|f\|_{\mathcal{H}}^2 \right\}, \qquad \lambda > 0,
$$
теорема о представителе гарантирует, что решение имеет вид $f^*(\mathbf{x}) = \sum_{j=1}^n \alpha_j K(\mathbf{x}, \mathbf{x}_j)$. Подставляя это представление и используя матрицу Грама $\mathbf{G}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$, получаем задачу для коэффициентов:
$$
\min_{\boldsymbol{\alpha} \in \mathbb{R}^n} \bigl\| \mathbf{y} - \mathbf{G}\boldsymbol{\alpha} \bigr\|_2^2 + \lambda \boldsymbol{\alpha}^\top \mathbf{G} \boldsymbol{\alpha}.
$$
Дифференцируя по $\boldsymbol{\alpha}$ и приравнивая к нулю, находим
$$
\boldsymbol{\alpha}^* = (\mathbf{G} + \lambda \mathbf{I})^{-1} \mathbf{y}.
$$

Теорема о представителе — краеугольный камень ядровых методов. Она сводит бесконечномерную задачу оптимизации над функциями к конечномерной задаче поиска коэффициентов $\boldsymbol{\alpha}$, причём размерность равна числу обучающих примеров, а не размерности спрямляющего пространства. Именно это позволяет работать с бесконечномерными пространствами признаков без явного построения $\phi$: решение всегда записывается через значения ядра на обучающих объектах. Несмотря на бесконечную ёмкость RKHS, регуляризация по норме $\|f\|_{\mathcal{H}}$ эффективно контролирует сложность, подавляя высокочастотные компоненты и предотвращая переобучение.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что означает «воспроизводящее» в названии RKHS? Поясните на примере: как вычислить значение функции $f$ в точке $\mathbf{x}$, используя только скалярное произведение?
2. Почему в определении RKHS требуется непрерывность функционала оценки? К чему приводит его отсутствие?
3. Опишите, как выглядит воспроизводящее ядро для линейного ядра $K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top \mathbf{y}$. Что является элементами пространства $\mathcal{H}$ и какова норма?
4. Сформулируйте теорему о представителе своими словами. Почему она так важна для практического использования ядровых методов?
5. Приведите пример функции, принадлежащей RKHS, и функции, не принадлежащей RKHS для линейного ядра. (Подсказка: рассмотрите константную функцию $f(\mathbf{x}) = 1$ на $\mathbb{R}^D$.)

**Для профессионалов:**

1. Докажите, что функция $K(\mathbf{x}, \mathbf{y}) = \langle K_{\mathbf{x}}, K_{\mathbf{y}} \rangle_{\mathcal{H}}$ является положительно определённым ядром, исходя из определения RKHS.
2. Сформулируйте и докажите теорему о представителе для случая квадратичной функции потерь $L(y, t) = (y-t)^2$ и регуляризатора $\Omega(\|f\|) = \lambda \|f\|_{\mathcal{H}}^2$. Выпишите явное выражение для коэффициентов $\boldsymbol{\alpha}$.
3. Покажите, что RKHS с RBF-ядром плотно в пространстве $L_2(\mathbb{R}^D)$ относительно $L_2$-нормы, но не совпадает с ним. Приведите конкретный пример $L_2$-функции, не принадлежащей этому RKHS.
4. Объясните связь между гладкостью функций в RKHS и скоростью убывания собственных чисел в разложении Мерсера. Какой вид имеет норма в терминах коэффициентов разложения по собственным функциям?
5. Приведите пример задачи, в которой решение, полученное с помощью ядрового метода, по теореме о представителе выражается через обучающие объекты, и объясните, почему это не приводит к переобучению, несмотря на то что размерность пространства может быть бесконечной. (Подсказка: роль регуляризатора.)



### 4. Построение сложных ядер: операции над ядрами и ядра для структурированных данных

В предыдущих разделах мы ввели несколько классических ядер — линейное, полиномиальное, RBF — и изучили их свойства. Однако реальные задачи машинного обучения часто требуют работать с данными, выходящими за рамки векторов $\mathbb{R}^D$: тексты, графы, биологические последовательности, изображения, вероятностные распределения. Возникает вопрос: как строить ядра для таких объектов и как комбинировать уже известные ядра, чтобы получить новые, более выразительные? Ответ дают алгебраические операции, сохраняющие положительную определённость, и специальные конструкции, учитывающие природу данных.

#### 4.1. Алгебраические операции, сохраняющие положительную определённость

Класс положительно определённых (ПО) ядер замкнут относительно ряда операций. Это позволяет конструировать сложные ядра из элементарных, подобно тому как из простых функций строят сложные.

*   **Сумма ядер.** Если $K_1$ и $K_2$ — ПО ядра на $\mathcal{X}$, то $K(\mathbf{x}, \mathbf{x}') = K_1(\mathbf{x}, \mathbf{x}') + K_2(\mathbf{x}, \mathbf{x}')$ также является ПО ядром. Доказательство непосредственно следует из определения: для любых $n$ точек и коэффициентов $c_i$
    $$
    \sum_{i,j} c_i c_j (K_1(\mathbf{x}_i, \mathbf{x}_j) + K_2(\mathbf{x}_i, \mathbf{x}_j)) = \sum_{i,j} c_i c_j K_1(\mathbf{x}_i, \mathbf{x}_j) + \sum_{i,j} c_i c_j K_2(\mathbf{x}_i, \mathbf{x}_j) \ge 0.
    $$

*   **Умножение на неотрицательную константу.** Для $c \ge 0$ ядро $c K$ является ПО.

*   **Произведение ядер (теорема Шура).** Если $K_1$ и $K_2$ — ПО ядра, то их поточечное произведение $K(\mathbf{x}, \mathbf{x}') = K_1(\mathbf{x}, \mathbf{x}') K_2(\mathbf{x}, \mathbf{x}')$ — также ПО ядро. Доказательство опирается на теорему Шура: поэлементное (адамарово) произведение двух неотрицательно определённых матриц является неотрицательно определённым. Действительно, для любого конечного набора точек матрицы Грама $\mathbf{G}_1$ и $\mathbf{G}_2$ положительно полуопределены; тогда их поэлементное произведение $\mathbf{G} = \mathbf{G}_1 \odot \mathbf{G}_2$ положительно полуопределено, что равносильно условию (2.1). Поскольку это выполнено для любого конечного набора, $K_1 K_2$ ПО. Отсюда автоматически следует, что любая целая положительная степень ПО ядра также является ПО ядром, а значит, и многочлены с неотрицательными коэффициентами от ПО ядер сохраняют положительную определённость.

*   **Предельный переход.** Если последовательность ПО ядер $\{K_n\}$ сходится поточечно к функции $K$, то $K$ также является ПО ядром. Неравенство (2.1) сохраняется при пределе. Этот факт позволяет строить ядра через бесконечные ряды (например, RBF через экспоненту от линейного ядра).

Эти операции образуют «алгебру ядер» и обосновывают корректность многих эвристических конструкций.

#### 4.2. Нормировка ядер и экспонента

На практике часто бывает удобно работать с **нормированным ядром**

$$
\tilde{K}(\mathbf{x}, \mathbf{x}') = \frac{K(\mathbf{x}, \mathbf{x}')}{\sqrt{K(\mathbf{x}, \mathbf{x}) \, K(\mathbf{x}', \mathbf{x}')}}.
$$

Такое ядро всегда принимает значение $1$ при $\mathbf{x} = \mathbf{x}'$ и $\tilde{K}(\mathbf{x}, \mathbf{x}') \in [-1, 1]$. Геометрически оно соответствует косинусу угла между образами $\phi(\mathbf{x})$ и $\phi(\mathbf{x}')$ в пространстве $\mathcal{H}$. Действительно, поскольку $\|\phi(\mathbf{x})\|_{\mathcal{H}}^2 = K(\mathbf{x}, \mathbf{x})$, имеем

$$
\tilde{K}(\mathbf{x}, \mathbf{x}') = \frac{\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}}}{\|\phi(\mathbf{x})\|_{\mathcal{H}} \|\phi(\mathbf{x}')\|_{\mathcal{H}}} = \cos \angle(\phi(\mathbf{x}), \phi(\mathbf{x}')).
$$

Нормированное ядро сохраняет положительную определённость: его можно представить как произведение трёх ПО ядер.

Другой важный пример — **экспонента от ядра**: если $K$ — ПО ядро, то $\exp(K)$ — ПО ядро. Доказательство: $\exp(K) = \sum_{m=0}^\infty K^m / m!$, где $K^m$ — $m$-кратное произведение ядра на себя (ПО по теореме Шура); сумма ПО ядер есть ПО ядро; поточечный предел ПО ядер — ПО ядро. Именно так RBF-ядро $\exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$ получается из линейного ядра $\mathbf{x}^\top \mathbf{x}'$ с помощью аффинного преобразования, знака и экспоненты.

#### 4.3. Ядра для структурированных данных

Многие прикладные задачи требуют сравнения не векторов, а объектов сложной структуры — строк, графов, множеств, деревьев. Идея состоит в том, чтобы определить на этих объектах ядро, которое измеряет сходство через количество общих подструктур или через отображение в гильбертово пространство.

**Ядра на множествах.** Пусть объектами являются подмножества некоторого универсального множества $\mathcal{U}$ (например, наборы тегов, множества ключевых слов). Простейшее ПО ядро — **ядро пересечения**:

$$
K(A, B) = |A \cap B|.
$$

Его можно представить как $\langle \phi(A), \phi(B) \rangle$, где $\phi(A)$ — бинарный вектор длины $|\mathcal{U}|$, содержащий $1$ для элементов, входящих в $A$. Более общо, можно взвешивать элементы, получая ядро $\sum_{u \in A \cap B} w_u$ с неотрицательными весами $w_u$. Для мультимножеств (гистограмм) используют **гистограммное ядро** — линейное ядро над векторами частот. Другой популярный вариант — **$\chi^2$-ядро** $K_{\chi^2}(A,B) = \sum_u \frac{2 h_A(u) h_B(u)}{h_A(u) + h_B(u)}$, часто применяемое в компьютерном зрении (bag-of-visual-words). Оно также является ПО.

**Графовое ядро.** Пусть даны два графа $G$ и $G'$. Естественно измерять их сходство по общим подграфам. Однако вычисление всех подграфов требует экспоненциального времени. Поэтому на практике используют **прогулочные ядра** (random walk kernels), которые учитывают все общие маршруты фиксированной длины. Формально, для графа строится матрица смежности $\mathbf{A}$ (возможно, с метками рёбер), и ядро определяется как

$$
K(G, G') = \sum_{m=0}^\infty \lambda_m \sum_{v,v'} \bigl[ \mathbf{A}^m \otimes \mathbf{A}'^m \bigr]_{v,v'},
$$

где $\otimes$ — тензорное произведение, а $\lambda_m$ — убывающие веса. При подходящих условиях эта сумма сходится и является ПО ядром. Такой подход успешно применяется в хемоинформатике для предсказания свойств молекул.

**Строковое ядро (string kernel).** Для текстовых или генетических последовательностей (строк над алфавитом $\Sigma$) популярно **ядро на подстроках** (спектральное ядро). Пусть $\phi_s(x)$ — число вхождений подстроки $s$ в строку $x$. Тогда

$$
K(x, y) = \sum_{s \in \Sigma^*} w_s \phi_s(x) \phi_s(y)
$$

является ПО ядром (сумма произведений линейных ядер). На практике ограничиваются подстроками ограниченной длины ($k$-мерами) или используют **mismatch-ядра**, допускающие до $m$ несовпадений. Для каждой $k$-граммы определяются все её варианты с не более чем $m$ заменами символов, и вклад в ядро дают все такие приблизительные совпадения. Это повышает устойчивость к шуму и мутациям в биологических последовательностях.

**Ядро взвешенного выравнивания (alignment kernel).** Для последовательностей разной длины можно определить ядро, суммируя экспоненты от сходства по всем возможным путям выравнивания (динамическое программирование). Такое ядро тесно связано с алгоритмом Нидлмана–Вунша и широко применяется в биоинформатике.

#### 4.4. Ядра, не удовлетворяющие условию Мерсера, но используемые на практике

Исторически в SVM и других ядровых методах иногда применяли **сигмоидальное ядро**

$$
K_{\text{sigmoid}}(\mathbf{x}, \mathbf{x}') = \tanh(\gamma \mathbf{x}^\top \mathbf{x}' + r),
$$

мотивированное аналогией с нейронными сетями. Однако, как уже обсуждалось, оно не является положительно определённым при всех параметрах — матрица Грама может иметь отрицательные собственные значения. Несмотря на это, на практике оптимизационные алгоритмы (например, SMO) могут сходиться, и модель иногда показывает приемлемое качество. Почему это возможно? SVM не требует строгой ПО для численной сходимости к *какому-то* решению, но при этом теряются гарантии единственности решения, геометрическая интерпретация через скалярное произведение в $\mathcal{H}$ и выпуклость задачи. Поэтому использование таких ядер — это эвристика, и в серьёзных приложениях рекомендуется выбирать ядра с доказанной ПО.

#### 4.5. Углублённый взгляд: ядро Фишера (Fisher kernel)

> **Для читателя с математической подготовкой.** Часто объекты описываются не векторами признаков, а распределениями — например, документ можно представить моделью LDA, а изображение — смесью гауссиан. Возникает вопрос: как построить ядро между такими вероятностными моделями? Элегантное решение даёт *ядро Фишера* (Fisher kernel), позволяющее использовать генеративные модели в дискриминативных алгоритмах вроде SVM.

Пусть $p(\mathbf{x} \mid \boldsymbol{\theta})$ — параметрическое семейство плотностей, и $\boldsymbol{\theta}$ — вектор параметров. Для наблюдения $\mathbf{x}$ определим **фишеровский скор** (Fisher score) — градиент логарифмической плотности по параметрам:

$$
\mathbf{U}_{\mathbf{x}} = \nabla_{\boldsymbol{\theta}} \log p(\mathbf{x} \mid \boldsymbol{\theta}).
$$

Информационная матрица Фишера $\mathbf{I}(\boldsymbol{\theta})$ задаётся как

$$
\mathbf{I}(\boldsymbol{\theta}) = \mathbb{E}_{\mathbf{x} \sim p(\cdot \mid \boldsymbol{\theta})} \bigl[ \mathbf{U}_{\mathbf{x}} \mathbf{U}_{\mathbf{x}}^\top \bigr].
$$

Тогда **ядро Фишера** определяется как

$$
K(\mathbf{x}, \mathbf{x}') = \mathbf{U}_{\mathbf{x}}^\top \mathbf{I}(\boldsymbol{\theta})^{-1} \mathbf{U}_{\mathbf{x}'}. \tag{4.1}
$$

Геометрический смысл: это скалярное произведение в пространстве параметров, индуцированное римановой метрикой Фишера. Мы вкладываем данные в «касательное пространство» статистического многообразия. Ядро является ПО, поскольку может быть записано как $\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle$ с $\phi(\mathbf{x}) = \mathbf{I}(\boldsymbol{\theta})^{-1/2} \mathbf{U}_{\mathbf{x}}$.

На практике информационную матрицу часто заменяют на единичную (получая простое *практическое ядро Фишера* — линейное ядро в пространстве score-векторов), либо оценивают по обучающей выборке. Fisher kernel широко применяется в биоинформатике (например, для классификации белковых последовательностей с моделями скрытых марковских цепей) и в задачах компьютерного зрения с генеративными моделями. Альтернативой служат **ядра на основе дивергенций** (например, $K(p, q) = \exp(-\alpha D_{\text{KL}}(p \| q))$), которые не всегда являются ПО, но удобны в байесовских постановках.

---

#### Вопросы для самопроверки

**Для начинающих:**

1.  Какие алгебраические операции сохраняют положительную определённость ядра? Приведите пример того, как из линейного ядра построить полиномиальное.
2.  Зачем нужно нормировать ядро? Как выглядит формула нормированного ядра и какой геометрический смысл она имеет?
3.  Что такое строковое ядро? Как можно сравнить две текстовые строки с помощью ядра, не преобразуя их в числовой вектор?
4.  Почему графовое ядро вычислять сложно? Какую аппроксимацию используют на практике?
5.  Приведите пример ядра, которое не является положительно определённым, но иногда используется в SVM. Какие риски с этим связаны?

**Для профессионалов:**

1.  Докажите, что экспонента от ПО ядра $\exp(K)$ является ПО ядром.
2.  Найдите условие на параметры $\gamma$ и $r$, при котором сигмоидальное ядро $\tanh(\gamma \mathbf{x}^\top \mathbf{y} + r)$ является ПО для данных, лежащих на единичной сфере. (Подсказка: используйте геометрическую интерпретацию и свойства угла.)
3.  Объясните, как построить ядро для сравнения двух множеств разной мощности. Предложите конструкцию и докажите её ПО.
4.  Сравните Fisher kernel и ядро, основанное на симметризованной дивергенции Кульбака–Лейблера. В чём преимущества и недостатки каждого?
5.  Предложите конструкцию ядра для деревьев (например, синтаксических деревьев разбора предложений). Опишите идею и докажите, что ваше ядро ПО.




### 5. Применение ядер в линейной регрессии: ядерный гребень (Kernel Ridge Regression)

В предыдущих разделах мы построили математический фундамент ядровых методов: положительно определённые ядра, теорему Мерсера, воспроизводящие ядерные гильбертовы пространства (RKHS) и теорему о представителе. Теперь мы применим этот аппарат к одной из самых простых и прозрачных моделей — гребневой регрессии (ridge regression), обобщив её на нелинейные зависимости с помощью ядер. Полученный метод называется **ядерным гребнем** (Kernel Ridge Regression, KRR) и служит идеальной отправной точкой для понимания более сложных ядровых алгоритмов, таких как SVM и гауссовские процессы.

---

#### 5.1. Вывод решения для KRR

Рассмотрим задачу регрессии. Дана обучающая выборка $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, где $\mathbf{x}_i \in \mathcal{X}$, $y_i \in \mathbb{R}$. Мы ищем функцию $f : \mathcal{X} \to \mathbb{R}$ в RKHS $\mathcal{H}$ с воспроизводящим ядром $K$, минимизирующую регуляризованный эмпирический риск с квадратичной функцией потерь:

$$
\min_{f \in \mathcal{H}} \left\{ \sum_{i=1}^n \bigl(y_i - f(\mathbf{x}_i)\bigr)^2 + \lambda \|f\|_{\mathcal{H}}^2 \right\}, \qquad \lambda > 0. \tag{5.1}
$$

Параметр $\lambda$ управляет компромиссом между точностью на обучающих данных и гладкостью (сложностью) функции, измеряемой нормой RKHS. Согласно теореме о представителе (раздел 3.5), решение задачи (5.1) существует и имеет вид конечной линейной комбинации ядровых функций, центрированных на обучающих точках:

$$
f^*(\mathbf{x}) = \sum_{j=1}^n \alpha_j K(\mathbf{x}, \mathbf{x}_j), \qquad \alpha_j \in \mathbb{R}. \tag{5.2}
$$

Подставим (5.2) в (5.1). Введём матрицу Грама $\mathbf{K} \in \mathbb{R}^{n \times n}$ с элементами $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$ и вектор коэффициентов $\boldsymbol{\alpha} = (\alpha_1, \dots, \alpha_n)^\top$. Тогда значения функции на обучающих точках запишутся как $\mathbf{f} = \mathbf{K} \boldsymbol{\alpha}$. Эмпирический риск примет вид $\|\mathbf{y} - \mathbf{K} \boldsymbol{\alpha}\|_2^2$, где $\mathbf{y} = (y_1, \dots, y_n)^\top$.

Регуляризационный член $\|f\|_{\mathcal{H}}^2$ для функций вида $f = \sum_{j=1}^n \alpha_j K(\cdot, \mathbf{x}_j)$ вычисляется через ядро: в силу воспроизводящего свойства,

$$
\|f\|_{\mathcal{H}}^2 = \sum_{i=1}^n \sum_{j=1}^n \alpha_i \alpha_j \langle K(\cdot, \mathbf{x}_i), K(\cdot, \mathbf{x}_j) \rangle_{\mathcal{H}} = \boldsymbol{\alpha}^\top \mathbf{K} \boldsymbol{\alpha}. \tag{5.3}
$$

Таким образом, бесконечномерная задача (5.1) сводится к конечномерной выпуклой оптимизации по $\boldsymbol{\alpha}$:

$$
\min_{\boldsymbol{\alpha} \in \mathbb{R}^n} \left\{ \|\mathbf{y} - \mathbf{K} \boldsymbol{\alpha}\|_2^2 + \lambda \boldsymbol{\alpha}^\top \mathbf{K} \boldsymbol{\alpha} \right\}. \tag{5.4}
$$

Функционал квадратичен по $\boldsymbol{\alpha}$. Приравнивая градиент к нулю, получаем линейную систему:

$$
-\mathbf{K}(\mathbf{y} - \mathbf{K} \boldsymbol{\alpha}) + \lambda \mathbf{K} \boldsymbol{\alpha} = \mathbf{0} \quad \Longrightarrow \quad (\mathbf{K}^2 + \lambda \mathbf{K}) \boldsymbol{\alpha} = \mathbf{K} \mathbf{y}.
$$

Если матрица $\mathbf{K}$ обратима (что обычно выполняется для строго ПО ядер при различных $\mathbf{x}_i$), то, умножая слева на $\mathbf{K}^{-1}$, получаем

$$
(\mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha} = \mathbf{y}. \tag{5.5}
$$

Таким образом, оптимальный вектор коэффициентов находится как

$$
\boldsymbol{\alpha}^* = (\mathbf{K} + \lambda \mathbf{I})^{-1} \mathbf{y}. \tag{5.6}
$$

Соответствующая функция предсказания:

$$
f^*(\mathbf{x}) = \sum_{i=1}^n \alpha_i^* K(\mathbf{x}, \mathbf{x}_i) = \mathbf{k}(\mathbf{x})^\top (\mathbf{K} + \lambda \mathbf{I})^{-1} \mathbf{y}, \tag{5.7}
$$

где $\mathbf{k}(\mathbf{x}) = (K(\mathbf{x}, \mathbf{x}_1), \dots, K(\mathbf{x}, \mathbf{x}_n))^\top$ — вектор ядровых значений между новым объектом и обучающими точками.

Вывод (5.6)–(5.7) полностью аналогичен обычной гребневой регрессии, где матрица Грама $\mathbf{K}$ заменяет матрицу признаков $\mathbf{X} \mathbf{X}^\top$. В частности, если взять линейное ядро $K(\mathbf{x}, \mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$, то $\mathbf{K} = \mathbf{X} \mathbf{X}^\top$, и KRR в точности совпадает с обычной гребневой регрессией в пространстве исходных признаков.

#### 5.2. Идея реализации и алгоритмические шаги

Хотя данное пособие не приводит исполняемого кода, полезно описать структуру реализации KRR, которая следует непосредственно из формул (5.6)–(5.7).

1. **Обучение (fit).** На вход подаются обучающая матрица признаков $\mathbf{X} \in \mathbb{R}^{n \times D}$ и вектор целевых значений $\mathbf{y}$.
   * Вычисляется ядровая матрица $\mathbf{K}$: $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$ для всех $i,j = 1,\dots,n$. Вычислительная сложность этого шага — $O(n^2 \cdot \text{cost}(K))$.
   * Решается линейная система $(\mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha} = \mathbf{y}$ относительно $\boldsymbol{\alpha}$. Прямое обращение требует $O(n^3)$ операций, однако на практике можно использовать разложение Холецкого (поскольку $\mathbf{K} + \lambda \mathbf{I}$ симметрична и положительно определена) или итерационные методы (сопряжённые градиенты), особенно для разреженных или быстро убывающих ядер.

2. **Предсказание (predict).** Для новых объектов $\mathbf{X}_{\text{new}} \in \mathbb{R}^{m \times D}$:
   * Вычисляется матрица перекрёстных ядер $\mathbf{K}_{\text{new}} \in \mathbb{R}^{m \times n}$: $(\mathbf{K}_{\text{new}})_{ij} = K(\mathbf{x}_{\text{new}, i}, \mathbf{x}_j)$.
   * Предсказания вычисляются как $\hat{\mathbf{y}} = \mathbf{K}_{\text{new}} \boldsymbol{\alpha}$.

Ключевое наблюдение: размерность матрицы $\mathbf{K}$ равна числу обучающих примеров $n$, а не числу признаков $D$ или размерности RKHS. Это одновременно и преимущество (мы можем работать с бесконечномерными пространствами), и узкое место — кубическая сложность по $n$ делает KRR неприменимым к очень большим выборкам без дополнительных аппроксимаций.

#### 5.3. Сравнение с линейной регрессией на нелинейных данных

Проиллюстрируем работу KRR на классическом примере — восстановлении синусоидальной зависимости. Пусть данные порождены моделью

$$
y = \sin(2\pi x) + \varepsilon, \qquad x \sim \text{Uniform}(0, 1), \quad \varepsilon \sim \mathcal{N}(0, 0.1^2).
$$

Линейная гребневая регрессия (соответствующая линейному ядру $K(x, x') = x x'$) способна строить только прямые линии. На интервале $[0,1]$ синусоида имеет существенно нелинейный характер: она проходит полный период, и никакая прямая не может адекватно описать такие данные. Коэффициент детерминации $R^2$ для линейной модели будет близок к нулю, а среднеквадратичная ошибка велика.

Теперь применим KRR с RBF-ядром $K(x, x') = \exp(-\gamma (x - x')^2)$. Благодаря неявному отображению в бесконечномерное пространство, где линейные функции соответствуют очень гибким нелинейным кривым в исходной области, модель прекрасно восстанавливает синусоиду. При правильно подобранных $\lambda$ и $\gamma$ предсказания практически совпадают с истинной функцией, а ошибка стремится к уровню шума $\sigma^2$. Таким образом, KRR демонстрирует способность улавливать сложные нелинейные закономерности, оставаясь линейной моделью в RKHS.

#### 5.4. Роль гиперпараметров $\lambda$ и $\gamma$

Поведение KRR критически зависит от двух гиперпараметров: параметра регуляризации $\lambda$ и параметра ядра $\gamma$ (для RBF).

* **Параметр регуляризации $\lambda$** контролирует штраф за сложность функции. При $\lambda \to 0$ модель стремится точно интерполировать обучающие точки: $\boldsymbol{\alpha} = \mathbf{K}^{-1} \mathbf{y}$ (если $\mathbf{K}$ обратима), что ведёт к идеальной подгонке под обучающие данные, но, как правило, к сильному переобучению и высокодисперсионным предсказаниям на новых точках. При $\lambda \to \infty$ коэффициенты $\boldsymbol{\alpha}$ стремятся к нулю, и модель вырождается в константу, близкую к среднему значению $y$. Оптимальное $\lambda$ находят по кросс-валидации: оно минимизирует ошибку на отложенных данных, балансируя смещение и дисперсию.

* **Параметр $\gamma$ RBF-ядра** определяет характерный масштаб близости точек. При очень маленьких $\gamma$ (широкое ядро) $K(x, x') \approx 1$ для всех точек, и модель склонна к недообучению: она порождает очень гладкие, почти линейные функции. При очень больших $\gamma$ (узкое ядро) $K(x, x')$ быстро убывает с расстоянием, и модель становится чрезвычайно локальной: каждая точка влияет лишь на свою ближайшую окрестность, что ведёт к переобучению — функция сильно осциллирует, проходя через каждую обучающую точку, но плохо обобщая. Иллюстрацией служит график предсказаний при разных $\gamma$: для $\gamma=0.1$ — плавная волна, не доходящая до пиков; для $\gamma=100$ — изрезанная кривая, точно попадающая в точки, но далёкая от истинной синусоиды между ними.

На практике $\lambda$ и $\gamma$ оптимизируют совместно, используя сеточный поиск с кросс-валидацией или байесовские методы оптимизации.

#### 5.5. Вычислительная сложность и проблема масштабирования

Основное узкое место KRR — кубическая сложность $O(n^3)$ решения системы $(\mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha} = \mathbf{y}$ и квадратичная сложность $O(n^2)$ хранения матрицы $\mathbf{K}$. Для $n \lesssim 10^4$ эти затраты приемлемы на современных компьютерах, однако при $n \gtrsim 10^5$ прямое применение KRR становится затруднительным.

Для преодоления этого ограничения разработан ряд приближённых методов.

* **Метод Найстрёма (Nyström approximation).** Выбирается $m \ll n$ опорных точек, и полная ядерная матрица аппроксимируется низкоранговым разложением $\mathbf{K} \approx \mathbf{K}_{nm} \mathbf{K}_{mm}^{-1} \mathbf{K}_{nm}^\top$, где $\mathbf{K}_{mm}$ — ядерная матрица на опорных точках, а $\mathbf{K}_{nm}$ — перекрёстная матрица. Это снижает сложность обучения до $O(n m^2 + m^3)$ и предсказания до $O(m)$ на объект.

* **Случайные признаки (Random Fourier Features, RFF).** Для стационарных ядер, представимых как преобразование Фурье неотрицательной меры (теорема Бохнера), можно построить случайные конечномерные отображения $\phi_{\text{random}} : \mathcal{X} \to \mathbb{R}^D$, такие что $K(\mathbf{x}, \mathbf{x}') \approx \phi_{\text{random}}(\mathbf{x})^\top \phi_{\text{random}}(\mathbf{x}')$. После этого задача сводится к обычной гребневой регрессии в $D$-мерном пространстве, где $D$ — число случайных признаков (обычно $D \ll n$). Сложность обучения $O(n D^2 + D^3)$, предсказания $O(D)$.

* **Итерационные методы.** Систему $(\mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha} = \mathbf{y}$ можно решать с помощью метода сопряжённых градиентов, не формируя матрицу $\mathbf{K}$ явно, если есть быстрый способ вычислять произведения $\mathbf{K} \mathbf{v}$ (например, с помощью быстрого преобразования Фурье для стационарных ядер на регулярных сетках или с помощью специальных структур данных). Это позволяет работать с $n$ до нескольких миллионов.

Выбор конкретной аппроксимации зависит от природы данных, вида ядра и требуемой точности. В любом случае, KRR остаётся одним из наиболее элегантных и теоретически обоснованных нелинейных регрессионных методов.

---

#### 5.6. Углублённый взгляд: связь с гауссовскими процессами и формула Вудбери

> **Для читателя с математической подготовкой.** KRR не только решает задачу минимизации эмпирического риска, но и имеет глубокую связь с байесовским выводом и гауссовскими процессами (GP). Рассмотрим вероятностную модель: пусть $f \sim \mathcal{GP}(0, K)$, т.е. $f$ — гауссовский процесс с нулевым средним и ковариационной функцией $K$, и наблюдения $y_i = f(\mathbf{x}_i) + \varepsilon_i$, где $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$. Тогда апостериорное среднее $f$ в точке $\mathbf{x}$ равно
> $$
> \mathbb{E}[f(\mathbf{x}) \mid \mathbf{X}, \mathbf{y}] = \mathbf{k}(\mathbf{x})^\top (\mathbf{K} + \sigma^2 \mathbf{I})^{-1} \mathbf{y},
> $$
> что в точности совпадает с предсказанием KRR (5.7) при $\lambda = \sigma^2$. Таким образом, KRR и GP с нулевым средним дают идентичные предсказания среднего; различие заключается лишь в том, что GP предоставляет полную апостериорную ковариацию (меру неопределённости).
>
> **Формула Вудбери.** В ряде случаев выгоднее работать в двойственном пространстве признаков. Если ядро представимо как $K(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^\top \phi(\mathbf{x}')$ с явным (возможно, конечномерным) $\phi$, то решение можно переписать в прямой форме. Используя матричное тождество Вудбери,
> $$
> (\mathbf{K} + \lambda \mathbf{I})^{-1} = \frac{1}{\lambda} \mathbf{I} - \frac{1}{\lambda^2} \boldsymbol{\Phi}^\top (\mathbf{I} + \frac{1}{\lambda} \boldsymbol{\Phi} \boldsymbol{\Phi}^\top)^{-1} \boldsymbol{\Phi},
> $$
> где $\boldsymbol{\Phi}$ — матрица признаков, можно получить выражение для весов $\mathbf{w}$ в пространстве RKHS: $\mathbf{w} = \boldsymbol{\Phi}^\top \boldsymbol{\alpha} = \boldsymbol{\Phi}^\top (\mathbf{K} + \lambda \mathbf{I})^{-1} \mathbf{y} = (\boldsymbol{\Phi}^\top \boldsymbol{\Phi} + \lambda \mathbf{I})^{-1} \boldsymbol{\Phi}^\top \mathbf{y}$, что есть обычное ridge-решение. Это подчёркивает эквивалентность прямой и двойственной форм.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Почему решение задачи KRR всегда можно записать в виде линейной комбинации ядер $f(\mathbf{x}) = \sum_{i=1}^n \alpha_i K(\mathbf{x}, \mathbf{x}_i)$? Какая теорема это гарантирует?
2. Как из системы $(\mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha} = \mathbf{y}$ найти коэффициенты $\boldsymbol{\alpha}$? Какие методы решения линейных систем вы знаете?
3. Как параметр $\gamma$ RBF-ядра влияет на гладкость предсказаний? Опишите, что происходит при $\gamma \to 0$ и при $\gamma \to \infty$.
4. Почему вычислительная сложность $O(n^3)$ является проблемой для KRR? При каком размере выборки она становится критичной?
5. В каких ситуациях KRR существенно лучше линейной гребневой регрессии? Приведите пример данных.

**Для профессионалов:**

1. Выведите решение KRR, используя формулу Вудбери, и покажите эквивалентность прямой и двойственной форм. (Указание: начните с $f(\mathbf{x}) = \phi(\mathbf{x})^\top \mathbf{w}$ и приравняйте к ядерному представлению.)
2. Сравните KRR и гауссовские процессы: в чём сходство и различие? При каких условиях их предсказания совпадают?
3. Объясните, как выбрать $\lambda$ с помощью $k$-фолд кросс-валидации. Какие метрики ошибки при этом минимизируют?
4. Реализуйте ускоренное предсказание с помощью аппроксимации Найстрёма и опишите, как изменится вычислительная сложность обучения и предсказания.
5. Докажите, что KRR с линейным ядром $K(\mathbf{x}, \mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$ в точности совпадает с обычной гребневой регрессией (Ridge). (Указание: выразите веса $\mathbf{w}$ через $\boldsymbol{\alpha}$.)

### 6. Теоретические аспекты: скорость сходимости, эффективная размерность и связь с регуляризацией

В предыдущих разделах мы изучили ядровой трюк, теорему Мерсера, конструкцию RKHS и применение ядер в гребневой регрессии (KRR). Однако остаётся фундаментальный вопрос: *почему* ядровые методы, оперируя в бесконечномерных пространствах, не переобучаются и дают хорошие обобщения? Ответ кроется в том, что регуляризация через норму RKHS эффективно ограничивает сложность модели, а свойства ядра (скорость убывания собственных чисел) определяют «эффективную размерность» задачи. В этом завершающем разделе мы формализуем эти идеи с помощью понятий эффективной размерности, радемахеровской сложности и связей с теорией гладкости, а также рассмотрим множественное ядро обучение (Multiple Kernel Learning) как естественное расширение ядровых методов.

---

#### 6.1. Эффективная размерность и след ядра

Рассмотрим ядерную гребневую регрессию с матрицей Грама $\mathbf{K}$ и параметром регуляризации $\lambda > 0$. В обычной линейной регрессии с $p$ признаками число эффективных параметров равно $p$, и оно же фигурирует в степенях свободы модели. В RKHS размерность может быть бесконечной, но регуляризация подавляет большинство направлений, оставляя лишь «эффективные». Мерой этого служит **эффективная размерность** (effective dimensionality), определяемая как

$$
\mathcal{N}(\lambda) = \operatorname{tr}\bigl((\mathbf{K} + \lambda \mathbf{I})^{-1} \mathbf{K}\bigr). \tag{6.1}
$$

Величина $\mathcal{N}(\lambda)$ имеет несколько интерпретаций:

* В байесовской трактовке (гауссовские процессы) она равна числу эффективных параметров, контролирующих апостериорное среднее.
* Она совпадает с суммой собственных значений оператора сглаживания: если $\mu_1 \ge \mu_2 \ge \dots \ge 0$ — собственные числа матрицы $\mathbf{K}$, то
  $$
  \mathcal{N}(\lambda) = \sum_{i=1}^n \frac{\mu_i}{\mu_i + \lambda}. \tag{6.2}
  $$
  Каждое слагаемое лежит в $[0,1)$ и показывает, насколько $i$-е направление «пропускается» регуляризацией: при $\mu_i \gg \lambda$ оно почти полностью сохраняется, при $\mu_i \ll \lambda$ — сильно подавляется.

Используя разложение Мерсера $K(\mathbf{x}, \mathbf{x}') = \sum_{k=1}^\infty \lambda_k \psi_k(\mathbf{x}) \psi_k(\mathbf{x}')$ (здесь собственные числа ядра обозначены $\lambda_k$, не путать с параметром регуляризации), можно показать, что для больших $n$ эмпирическая эффективная размерность приближает

$$
\mathcal{N}(\lambda) \approx \sum_{k=1}^\infty \frac{\lambda_k}{\lambda_k + \lambda}. \tag{6.3}
$$

Таким образом, $\mathcal{N}(\lambda)$ напрямую связана со спектром ядра. Для ядер с быстрым убыванием собственных чисел (например, RBF, где $\lambda_k$ убывают экспоненциально) эффективная размерность мала даже при очень малых $\lambda$. Это объясняет, почему KRR с RBF-ядром может успешно обучаться на малых выборках, не переобучаясь: сложность модели эффективно конечна.

Верхняя граница: $\mathcal{N}(\lambda) \le \frac{\operatorname{tr}(\mathbf{K})}{\lambda}$. Поскольку $\operatorname{tr}(\mathbf{K}) = \sum_i \mu_i$, для ограниченных ядер (например, $\max_x K(x,x) \le \kappa$) след растёт линейно с $n$, и эффективная размерность имеет порядок $O(1/\lambda)$.

#### 6.2. Границы обобщения для ядровых методов

Рассмотрим задачу регрессии или классификации с ограниченной функцией потерь. Пусть $\mathcal{H}$ — RKHS с ядром $K$, и мы рассматриваем класс моделей с ограниченной нормой:

$$
\mathcal{F}_R = \{ f \in \mathcal{H} : \|f\|_{\mathcal{H}} \le R \}.
$$

Для получения гарантий обобщения часто используют **радемахеровскую сложность** (Rademacher complexity). Для выборки $\mathbf{x}_1, \dots, \mathbf{x}_n$ эмпирическая радемахеровская сложность класса $\mathcal{F}_R$ ограничена как

$$
\hat{\mathfrak{R}}_n(\mathcal{F}_R) \le \frac{R}{n} \sqrt{\operatorname{tr}(\mathbf{K})} = \frac{R}{n} \sqrt{\sum_{i=1}^n K(\mathbf{x}_i, \mathbf{x}_i)}. \tag{6.4}
$$

Это оценка показывает, что сложность класса растёт с радиусом шара $R$ и с величиной следа ядерной матрицы, но убывает с ростом $n$. Если ядро ограничено ($K(\mathbf{x}, \mathbf{x}) \le \kappa$), то $\hat{\mathfrak{R}}_n(\mathcal{F}_R) \le R \sqrt{\kappa / n}$.

С использованием этой оценки и стандартных результатов теории обобщения (например, теоремы Бартлетта–Мендельсона) получается следующая гарантия: для любого $f \in \mathcal{F}_R$ с вероятностью не менее $1 - \delta$ истинный риск $R(f)$ ограничен эмпирическим риском $\hat{R}_{\text{emp}}(f)$ как

$$
R(f) \le \hat{R}_{\text{emp}}(f) + \frac{2R}{n} \sqrt{\operatorname{tr}(\mathbf{K})} + 3\sqrt{\frac{\log(2/\delta)}{2n}}. \tag{6.5}
$$

Ключевое наблюдение: размерность RKHS в этой оценке не участвует явно — её заменяет след ядерной матрицы, который может быть намного меньше истинной размерности. Для RBF-ядра $\operatorname{tr}(\mathbf{K}) = n$, так как $K(\mathbf{x}_i, \mathbf{x}_i) = 1$, и оценка совпадает с той, что была бы для конечномерного пространства размерности $\approx n$. Однако более тонкие оценки используют эффективную размерность $\mathcal{N}(\lambda)$ и спектр ядра, что даёт значительно более узкие границы для быстро убывающих собственных чисел.

#### 6.3. Регуляризация и ранняя остановка в ядровых методах

Мы уже видели, что явная регуляризация через штраф $\lambda \|f\|_{\mathcal{H}}^2$ эквивалентна добавлению $\lambda \mathbf{I}$ к матрице $\mathbf{K}$. Альтернативный способ контроля сложности — **ранняя остановка** (early stopping) итерационных алгоритмов. Рассмотрим минимизацию эмпирического риска $\hat{R}_{\text{emp}}(f) = \frac{1}{n} \sum_i (y_i - f(\mathbf{x}_i))^2$ градиентным спуском в RKHS. Если начать с $f_0 = 0$ и выполнять шаги градиентного спуска, то $t$-я итерация $f_t$ лежит в подпространстве, натянутом на $K(\cdot, \mathbf{x}_i)$, и может быть записана как $f_t = \sum_i \alpha_i^{(t)} K(\cdot, \mathbf{x}_i)$. Доказано, что ранняя остановка эквивалентна добавлению регуляризации: чем меньше число итераций $t$, тем сильнее подавлены компоненты, соответствующие малым собственным числам матрицы $\mathbf{K}$. Фактически, роль $\lambda$ играет величина $1/t$.

Аналогичная связь существует для метода сопряжённых градиентов и других итерационных схем. Этот взгляд полезен при больших $n$: вместо явного обращения матрицы можно выполнить несколько десятков итераций градиентного спуска, что значительно быстрее и даёт естественную регуляризацию.

#### 6.4. Эмпирический выбор ядра и множественное ядро обучение (Multiple Kernel Learning)

На практике выбор ядра существенно влияет на качество модели. Часто априори неизвестно, какое ядро оптимально. Один из подходов — перебор по сетке и кросс-валидация. Более систематический метод — **множественное ядро обучение** (Multiple Kernel Learning, MKL). В MKL строится ядро как выпуклая комбинация базовых ядер:

$$
K(\mathbf{x}, \mathbf{x}') = \sum_{m=1}^M \beta_m K_m(\mathbf{x}, \mathbf{x}'), \qquad \beta_m \ge 0, \quad \sum_m \beta_m = 1,
$$

или более общо с дополнительной нормировкой. Параметры $\beta_m$ и коэффициенты модели (например, $\boldsymbol{\alpha}$ в SVM) оптимизируются совместно. Задача часто сводится к выпуклой оптимизации — например, к решению задачи полуопределённого программирования (SDP) или к эффективной чередующейся минимизации.

MKL позволяет автоматически определять релевантные представления данных (например, комбинировать ядра на различных признаковых подпространствах или ядра разных типов для гетерогенных данных) и повышает интерпретируемость — большие веса $\beta_m$ указывают на наиболее информативные модальности.

#### 6.5. Углублённый взгляд: неравенство Кона–Штайна и связь гладкости с RKHS

> **Для читателя с математической подготовкой.** Почему KRR с RBF-ядром отлично работает для гладких функций? Ответ даёт анализ в частотной области. Для RBF-ядра $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$ собственные функции в разложении Мерсера на $\mathbb{R}^D$ — это гармонические функции, а собственные числа убывают экспоненциально: $\lambda_k \sim \exp(-c k^{2/D})$. Следовательно, RKHS состоит из функций, чьи коэффициенты разложения $a_k$ по этому базису удовлетворяют $\sum_k a_k^2 / \lambda_k < \infty$. Это означает, что высокочастотные компоненты $a_k$ должны быть экспоненциально малы. Иными словами, функции из RKHS имеют экспоненциально затухающее преобразование Фурье и поэтому бесконечно гладкие (аналитические).
>
> **Неравенство Кона–Штайна** для ядер формализует эту связь: если $f \in \mathcal{H}$ с $\|f\|_{\mathcal{H}} \le R$, то её преобразование Фурье $\hat{f}$ удовлетворяет
> $$
> \int_{\mathbb{R}^D} |\hat{f}(\boldsymbol{\omega})|^2 \exp\!\Bigl(\frac{\|\boldsymbol{\omega}\|^2}{4\gamma}\Bigr) d\boldsymbol{\omega} \le R^2.
> $$
> Отсюда видно, что энергия $\hat{f}$ на высоких частотах экспоненциально подавлена. Это объясняет, почему KRR с RBF-ядром эффективно восстанавливает гладкие функции и избегает переобучения: штраф $\|f\|_{\mathcal{H}}^2$ в точности контролирует эту взвешенную энергию Фурье, подавляя негладкие компоненты.
>
> Таким образом, ядровые методы реализуют регуляризацию, согласованную с априорными предположениями о гладкости, закодированными в выборе ядра. Для RBF это предположение о бесконечной дифференцируемости; для ядер Матерна — о конечной гладкости и т.д. Спектральный анализ даёт точный количественный язык для описания этих свойств и построения оптимальных скоростей сходимости.

---

#### Вопросы для самопроверки (по всей теме ядер)

**Для начинающих:**

1. Что такое эффективная размерность в ядровых методах? Как она связана с собственными числами ядерной матрицы?
2. Почему радемахеровская сложность важна для понимания обобщающей способности? Как она оценивается для RKHS?
3. Что такое множественное ядро обучение (MKL) и какую проблему оно решает?
4. Почему ядровые методы хорошо работают для гладких функций? Приведите интуитивное объяснение на основе RBF-ядра.
5. Как выбрать подходящее ядро на практике? Какие факторы влияют на этот выбор?
6. Чем KRR отличается от SVM в контексте регрессии? В каких случаях стоит предпочесть один метод другому?

**Для профессионалов:**

1. Выведите границу обобщения для KRR через радемахеровскую сложность класса $\{f \in \mathcal{H} : \|f\|_{\mathcal{H}} \le R\}$. Укажите, какие предположения используются.
2. Докажите, что эффективная размерность $\mathcal{N}(\lambda)$ удовлетворяет неравенству $\mathcal{N}(\lambda) \le \frac{\operatorname{tr}(\mathbf{K})}{\lambda}$. При каком условии это неравенство близко к равенству?
3. Объясните, как задача множественного ядерного обучения (MKL) сводится к полуопределённому программированию (SDP). Какие алгоритмы решают эту задачу на практике?
4. Сравните скорость сходимости KRR для ядер с разным спектром: степенным убыванием (например, ядро Матерна) и экспоненциальным убыванием (RBF). Как это влияет на минимаксные скорости?
5. Приведите пример задачи, где KRR с RBF-ядром существенно превосходит SVM с тем же ядром, и пример обратной ситуации. Объясните причины различий.